In [62]:
import glob
import json
import pandas as pd
from datasets import load_dataset
import re
import numpy as np
from tqdm.auto import tqdm

# GSM8K

In [18]:
def extract_gold_answer(gold_text: str) -> str:
    """Extract the ground-truth answer from a GSM8K solution string.
    GSM8K solutions end with '#### <answer>'.
    """
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", gold_text)
    if not match:
        raise ValueError(f"No '####' answer found in gold text: {gold_text!r}")
    return match.group(1).replace(",", "")

In [23]:
def extract_predicted_answer(model_output: str) -> str | None:
    """Extract the model's predicted final answer from free-form output.
    Tries, in order:
      1. '#### <answer>' (if the model was prompted to use this format)
      2. 'The answer is <answer>'
      3. The last standalone number in the text (fallback)
    Returns None if nothing could be parsed.
    """
    # 1. Prefer explicit #### delimiter
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", model_output)
    if match:
        return match.group(1).replace(",", "")

    # 2. "The answer is X" phrasing
    match = re.search(
        r"[Tt]he answer is\s*\$?(-?[\d,]+(?:\.\d+)?)", model_output
    )
    if match:
        return match.group(1).replace(",", "")

    # 3. Fallback: last number in the output
    numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", model_output)
    if numbers:
        return numbers[-1].replace(",", "")

    return None

In [26]:
def normalize_number(value: str) -> float | None:
    """Convert a string to a float for comparison. Returns None on failure."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def is_correct(predicted: str | None, gold: str) -> bool:
    pred_num = normalize_number(predicted)
    gold_num = normalize_number(gold)
    if pred_num is None or gold_num is None:
        return False
    return abs(pred_num - gold_num) < 1e-4

In [47]:
response_files = glob.glob("generations_gsm8k/*/*")

In [14]:
dataset = load_dataset("openai/gsm8k", "main")["test"]

In [19]:
labels = [extract_gold_answer(ans) for ans in dataset["answer"]]

In [48]:
response_record = []
for r in response_files:
    responses = json.load(open(r, "r"))
    splits = r.split("/")
    model = splits[1]
    persona = splits[2].replace("_", " ")[:-5]
    response_record.extend([{"model": model, "persona": persona, "response": y, "label": l, "question_id": idx} for idx, (y, l) in enumerate(zip(responses, labels))])
df = pd.DataFrame(response_record)

In [49]:
df["extracted_answer"] = df.response.apply(lambda x: extract_predicted_answer(x))

In [51]:
df["correct"] = df.apply(lambda x: is_correct(x["extracted_answer"], x["label"]), axis=1)

In [77]:
df

,model,persona,response,label,question_id,extracted_answer,correct
0,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",18,0,18,True
1,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",3,1,3,True
2,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a seeker of numbers, a mind that seeks to ...",70000,2,-110000,False
3,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of measurement, a mere flicker ...",540,3,540,True
4,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",20,4,30,False
...,...,...,...,...,...,...,...
166189,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 2,8,1314,2,False
166190,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 1,5,1315,1,False
166191,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 240,230,1316,240,False
166192,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,Let the number of chickens be \(c\) and the nu...,5,1317,5,True


In [52]:
df.groupby("model").correct.mean()

model
NVIDIA-Nemotron-3-Nano-4B-BF16              0.240
NVIDIA-Nemotron-3-Nano-4B-BF16-SFT+DPO-v2   0.511
Qwen3.5-4B                                  0.759
Qwen3.5-4B-SFT+DPO-v2                       0.851
gemma-4-12B-it                              0.947
gemma-4-12B-it-SFT+DPO-v2                   0.933
Name: correct, dtype: float64

In [53]:
df[df.model.str.contains("NVIDIA")].groupby(["model", "persona"]).correct.mean()

model                                      persona                           
NVIDIA-Nemotron-3-Nano-4B-BF16             Ancient Manipulative Vampire         0.250
                                           Bubbly Baker                         0.286
                                           Criminal Law Professor               0.230
                                           Cunning Cyber Mercenary              0.182
                                           Doting Grandmother                   0.304
                                           Dramatic Romance Novelist            0.335
                                           Enthusiastic Kindergarten Teacher    0.331
                                           Fanatical Cult Leader                0.216
                                           Forensic Pathologist                 0.240
                                           Hardened Bank Robber                 0.213
                                           Hollywood VFX Super

In [54]:
def normalize_number(value) -> float | None:
    """Convert a value to float for comparison, handling strings/commas."""
    if pd.isna(value):
        return None
    try:
        return float(str(value).replace(",", "").strip())
    except (TypeError, ValueError):
        return None


def compute_correctness(df: pd.DataFrame) -> pd.DataFrame:
    """Add a boolean 'correct' column based on label vs extracted_answer."""
    df = df.copy()
    df["label_num"] = df["label"].apply(normalize_number)
    df["pred_num"] = df["extracted_answer"].apply(normalize_number)
    df["correct"] = (
        df["label_num"].notna()
        & df["pred_num"].notna()
        & (np.abs(df["label_num"] - df["pred_num"]) < 1e-4)
    )
    return df


def split_base_and_variant(model_name: str) -> str:
    """Strip training-run suffixes to get the base model family name.
    E.g. 'Qwen3.5-4B-SFT+DPO-v2' -> 'Qwen3.5-4B'
         'Qwen3.5-4B-BF16'       -> 'Qwen3.5-4B'
    Adjust the pattern if your naming convention differs.
    """
    return re.sub(r"-(SFT\+DPO-v\d+|BF16)$", "", model_name)


def build_persona_gap_table(df: pd.DataFrame) -> pd.DataFrame:
    df = compute_correctness(df)
    df["is_trained"] = df["model"].str.contains(r"SFT\+DPO", regex=True)
    df["base_family"] = df["model"].apply(split_base_and_variant)

    # Per (base_family, persona, trained/untrained) accuracy
    acc = (
        df.groupby(["base_family", "persona", "is_trained"])["correct"]
        .mean()
        .reset_index()
    )

    pivot = acc.pivot_table(
        index=["base_family", "persona"],
        columns="is_trained",
        values="correct",
    ).reset_index()

    pivot = pivot.rename(columns={False: "acc_before", True: "acc_after"})
    pivot = pivot.dropna(subset=["acc_before", "acc_after"])  # need both to compare

    pivot["gap"] = pivot["acc_after"] - pivot["acc_before"]
    pivot["abs_gap"] = pivot["gap"].abs()

    # Also attach sample sizes for sanity-checking small-n noise
    counts = (
        df.groupby(["base_family", "persona", "is_trained"])["correct"]
        .size()
        .reset_index(name="n")
        .pivot_table(
            index=["base_family", "persona"], columns="is_trained", values="n"
        )
        .reset_index()
        .rename(columns={False: "n_before", True: "n_after"})
    )
    pivot = pivot.merge(counts, on=["base_family", "persona"], how="left")

    return pivot.sort_values("abs_gap", ascending=False)


def top_gaps(df: pd.DataFrame, n: int = 20, min_samples: int = 30) -> pd.DataFrame:
    """Biggest per-model-family, per-persona accuracy gaps, filtered for
    reasonable sample size so you're not chasing noise from tiny buckets."""
    gap_table = build_persona_gap_table(df)
    gap_table = gap_table[
        (gap_table["n_before"] >= min_samples) & (gap_table["n_after"] >= min_samples)
    ]
    return gap_table.head(n)[
        ["base_family", "persona", "acc_before", "acc_after", "gap", "n_before", "n_after"]
    ]

result = top_gaps(df, n=20, min_samples=30)
pd.set_option("display.float_format", "{:.3f}".format)
print(result.to_string(index=False))

   base_family                           persona  acc_before  acc_after    gap  n_before  n_after
    Qwen3.5-4B            Criminal Law Professor       0.033      0.913  0.879  1319.000 1319.000
    Qwen3.5-4B              Forensic Pathologist       0.422      0.902  0.480  1319.000 1319.000
    Qwen3.5-4B             Fanatical Cult Leader       0.589      0.762  0.173  1319.000 1319.000
    Qwen3.5-4B          Trendy Beauty Influencer       0.914      0.784 -0.130  1319.000 1319.000
    Qwen3.5-4B         Dramatic Romance Novelist       0.635      0.759  0.124  1319.000 1319.000
    Qwen3.5-4B      Ancient Manipulative Vampire       0.674      0.783  0.109  1319.000 1319.000
    Qwen3.5-4B                Military Historian       0.847      0.926  0.080  1319.000 1319.000
    Qwen3.5-4B Enthusiastic Kindergarten Teacher       0.751      0.825  0.074  1319.000 1319.000
gemma-4-12B-it             Fanatical Cult Leader       0.923      0.876 -0.048  1319.000 1319.000
gemma-4-12B-it      

In [55]:
REFUSAL_PATTERNS = [
    r"\bi can'?t\b",
    r"\bi cannot\b",
    r"\bi won'?t\b",
    r"\bi'?m not able to\b",
    r"\bi'?m unable to\b",
    r"\bas an ai\b",
    r"\bi refuse\b",
    r"\bi shall not\b",
    r"\bi will not\b",
    r"\blet'?s not\b",
    r"\bi don'?t (do|answer|engage with)\b",
    r"\bbeneath (me|my)\b",
    r"\bnot (something|a task) i\b",
]
REFUSAL_REGEX = re.compile("|".join(REFUSAL_PATTERNS), re.IGNORECASE)


def normalize_number(value):
    if pd.isna(value):
        return None
    try:
        return float(str(value).replace(",", "").strip())
    except (TypeError, ValueError):
        return None


def looks_like_refusal(response: str) -> bool:
    if not isinstance(response, str):
        return False
    return bool(REFUSAL_REGEX.search(response))


def annotate(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["label_num"] = df["label"].apply(normalize_number)
    df["pred_num"] = df["extracted_answer"].apply(normalize_number)
    df["correct"] = (
        df["label_num"].notna()
        & df["pred_num"].notna()
        & (np.abs(df["label_num"] - df["pred_num"]) < 1e-4)
    )
    df["refused"] = df["response"].apply(looks_like_refusal)
    df["is_trained"] = df["model"].str.contains(r"SFT\+DPO", regex=True)
    df["base_family"] = df["model"].str.replace(
        r"-(SFT\+DPO-v\d+|BF16)$", "", regex=True
    )
    return df


def refusal_rate_table(df: pd.DataFrame) -> pd.DataFrame:
    """Refusal rate per model, sorted descending."""
    df = annotate(df)
    return (
        df.groupby("model")["refused"]
        .agg(refusal_rate="mean", n="size")
        .sort_values("refusal_rate", ascending=False)
    )


def refusal_gap_by_persona(df: pd.DataFrame, min_samples: int = 20) -> pd.DataFrame:
    """Per (base_family, persona): refusal rate before vs after training."""
    df = annotate(df)

    rates = (
        df.groupby(["base_family", "persona", "is_trained"])["refused"]
        .agg(refusal_rate="mean", n="size")
        .reset_index()
    )

    pivot = rates.pivot_table(
        index=["base_family", "persona"],
        columns="is_trained",
        values=["refusal_rate", "n"],
    )
    pivot.columns = [f"{a}_{'after' if b else 'before'}" for a, b in pivot.columns]
    pivot = pivot.reset_index().dropna(
        subset=["refusal_rate_before", "refusal_rate_after"]
    )

    pivot = pivot[
        (pivot["n_before"] >= min_samples) & (pivot["n_after"] >= min_samples)
    ]

    pivot["refusal_gap"] = pivot["refusal_rate_after"] - pivot["refusal_rate_before"]
    pivot["abs_refusal_gap"] = pivot["refusal_gap"].abs()

    return pivot.sort_values("abs_refusal_gap", ascending=False)[
        [
            "base_family",
            "persona",
            "refusal_rate_before",
            "refusal_rate_after",
            "refusal_gap",
            "n_before",
            "n_after",
        ]
    ]


def error_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    """For each model, split wrong answers into: refusal vs genuine math error."""
    df = annotate(df)
    wrong = df[~df["correct"]]

    breakdown = (
        wrong.groupby("model")
        .agg(
            total_wrong=("correct", "size"),
            refusals=("refused", "sum"),
        )
        .reset_index()
    )
    breakdown["math_errors"] = breakdown["total_wrong"] - breakdown["refusals"]
    breakdown["pct_of_errors_are_refusals"] = (
        breakdown["refusals"] / breakdown["total_wrong"]
    )
    return breakdown.sort_values("pct_of_errors_are_refusals", ascending=False)



print("=== Refusal rate by model ===")
print(refusal_rate_table(df))

print("\n=== Biggest refusal-rate gaps by persona (before vs after training) ===")
print(refusal_gap_by_persona(df, min_samples=20).head(20).to_string(index=False))

print("\n=== Of the wrong answers, what fraction are refusals vs math errors? ===")
print(error_breakdown(df))

=== Refusal rate by model ===
                                           refusal_rate      n
model                                                         
Qwen3.5-4B                                        0.115  27699
Qwen3.5-4B-SFT+DPO-v2                             0.012  27699
gemma-4-12B-it                                    0.010  27699
NVIDIA-Nemotron-3-Nano-4B-BF16-SFT+DPO-v2         0.002  27699
gemma-4-12B-it-SFT+DPO-v2                         0.001  27699
NVIDIA-Nemotron-3-Nano-4B-BF16                    0.000  27699

=== Biggest refusal-rate gaps by persona (before vs after training) ===
   base_family                      persona  refusal_rate_before  refusal_rate_after  refusal_gap  n_before  n_after
    Qwen3.5-4B       Criminal Law Professor                0.672               0.001       -0.671  1319.000 1319.000
    Qwen3.5-4B         Forensic Pathologist                0.647               0.001       -0.646  1319.000 1319.000
    Qwen3.5-4B   Sleazy Corporate Embezzle

In [56]:
df

,model,persona,response,label,question_id,extracted_answer,correct
0,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",18,0,18,True
1,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",3,1,3,True
2,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a seeker of numbers, a mind that seeks to ...",70000,2,-110000,False
3,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of measurement, a mere flicker ...",540,3,540,True
4,Qwen3.5-4B-SFT+DPO-v2,Fanatical Cult Leader,"Ah, a question of numbers, a mere flicker in t...",20,4,30,False
...,...,...,...,...,...,...,...
166189,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 2,8,1314,2,False
166190,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 1,5,1315,1,False
166191,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,#### 240,230,1316,240,False
166192,NVIDIA-Nemotron-3-Nano-4B-BF16,Hollywood VFX Supervisor,Let the number of chickens be \(c\) and the nu...,5,1317,5,True


In [78]:
def paired_bootstrap_ci(
    df,
    model_a,
    model_b,
    unit_cols=("persona", "question_id"),
    metric_col="correct",
    n_boot=10_000,
    ci=0.95,
    seed=42,
):
    """
    Bootstrap the paired difference in accuracy:
        accuracy(model_b) - accuracy(model_a)

    Returns:
        observed difference, lower CI, upper CI
    """
    rng = np.random.default_rng(seed)

    a = (
        df[df["model"] == model_a]
        .set_index(list(unit_cols))[metric_col]
    )
    b = (
        df[df["model"] == model_b]
        .set_index(list(unit_cols))[metric_col]
    )

    # Keep only units evaluated by both models
    paired = pd.concat(
        [a.rename("a"), b.rename("b")],
        axis=1,
        join="inner",
    ).dropna()

    differences = paired["b"].astype(float) - paired["a"].astype(float)

    observed = differences.mean()

    # Resample paired observations
    boot_means = np.empty(n_boot)

    n = len(differences)

    for i in range(n_boot):
        sample = rng.choice(differences.values, size=n, replace=True)
        boot_means[i] = sample.mean()

    alpha = 1 - ci

    lower = np.quantile(boot_means, alpha / 2)
    upper = np.quantile(boot_means, 1 - alpha / 2)

    return observed, lower, upper

In [129]:
rng = np.random.default_rng(42)

a = (
    df[df["model"] == "gemma-4-12B-it"]
    .set_index(list(("persona", "question_id")))["correct"]
)
b = (
    df[df["model"] == "gemma-4-12B-it-SFT+DPO-v2"]
    .set_index(list(("persona", "question_id")))["correct"]
)

In [84]:
model_families = [
    {
        "name": "Nemotron",
        "base": "NVIDIA-Nemotron-3-Nano-4B-BF16",
        "trained": "NVIDIA-Nemotron-3-Nano-4B-BF16-SFT+DPO-v2",
    },
    {
        "name": "Qwen",
        "base": "Qwen3.5-4B",
        "trained": "Qwen3.5-4B-SFT+DPO-v2",
    },
    {
        "name": "Gemma",
        "base": "gemma-4-12B-it",
        "trained": "gemma-4-12B-it-SFT+DPO-v2",
    },
]

results = []

for family in model_families:
        diff, lower, upper = paired_bootstrap_ci(
            df,
            family["base"],
            family["trained"],
        )

        results.append({
            "model_family": family["name"],
            "impact": diff,
            "ci_lower": lower,
            "ci_upper": upper,
        })

results_df = pd.DataFrame(results)

In [85]:
results_df

,model_family,impact,ci_lower,ci_upper
0,Nemotron,0.2711,0.2637,0.2785
1,Qwen,0.0922,0.0867,0.0980
2,Gemma,-0.0138,-0.0166,-0.0109
